### 3.4 Проверка гипотез на train

**Входы**

- train и гипотезы, оставшиеся после попарного анализа

**Выходы**

- Результаты проверки гипотез и обновлённая таблица выводов

**Что делаем**

Сравним группы по сочетаниям признаков и проверим предположения, для которых недостаточно попарных коэффициентов

#### Подготовка групп

Группы для проверок сформируем по train

In [ ]:
analysis_train = train.copy()
analysis_train['grade_ordered'] = pd.Categorical(analysis_train['loan_grade'], categories=list('ABCDEFG'), ordered=True)
analysis_train['income_quartile'] = pd.qcut(analysis_train['person_income'], 4, duplicates='drop')
analysis_train['amount_quartile'] = pd.qcut(analysis_train['loan_amnt'], 4, duplicates='drop')
analysis_train['age_band'] = pd.cut(analysis_train['person_age'], [18, 25, 35, 50, 100], include_lowest=True)
analysis_train['history_band'] = pd.cut(analysis_train['cb_person_cred_hist_length'], [0, 5, 10, 15, 20, 40], include_lowest=True)
analysis_train['file_block'] = pd.cut(analysis_train.index, [0, 8000, 16000, 24000, len(data_raw)], include_lowest=True)


def default_summary(frame, groups):
    return frame.groupby(groups, observed=True)['loan_status'].agg(n='size', defaults='sum', default_rate='mean')

def group_summary_view(table):
    return table.round(4).rename(columns={
        'n': 'Записей', 'defaults': 'Дефолтов', 'default_rate': 'Доля дефолтов',
        'rate_n': 'Известных ставок', 'rate_median': 'Медиана ставки',
        'rate_q25': 'Ставка: 25%', 'rate_q75': 'Ставка: 75%',
        'history_groups': 'Групп по длине истории', 'min_cell_n': 'Минимум записей в группе',
        'min_default_rate': 'Наименьшая доля дефолтов', 'max_default_rate': 'Наибольшая доля дефолтов',
        'age_median': 'Медиана возраста', 'history_median': 'Медиана длины истории',
        'income_median': 'Медиана дохода', 'grade_A_share': 'Доля грейда A',
        'rate_missing': 'Доля пропусков ставки',
    })


#### 3.4.1 Грейд и ставка

Сравним распределение ставки и частоту дефолта по грейдам. Дополнительно разобьём ставки внутри каждого грейда на три группы

In [ ]:
grade_summary = analysis_train.groupby('grade_ordered', observed=True).agg(
    n=('loan_status', 'size'), default_rate=('loan_status', 'mean'),
    rate_n=('loan_int_rate', 'count'), rate_median=('loan_int_rate', 'median'),
    rate_q25=('loan_int_rate', lambda s: s.quantile(.25)),
    rate_q75=('loan_int_rate', lambda s: s.quantile(.75)),
)
# Внутри каждого грейда — свои терцили ставки: масштабы между грейдами не смешиваем.
analysis_train['rate_within_grade'] = analysis_train.groupby('grade_ordered', observed=True)['loan_int_rate'].transform(
    lambda s: pd.qcut(s, 3, labels=False, duplicates='drop')
)
within_grade_rate_summary = default_summary(analysis_train.dropna(subset=['rate_within_grade']), ['grade_ordered', 'rate_within_grade'])
display(group_summary_view(grade_summary))
display(group_summary_view(within_grade_rate_summary))


#### 3.4.2 Доля кредита от дохода

Проверим, насколько исходная доля соответствует отношению суммы кредита к доходу

In [ ]:
ratio_error = (analysis_train['loan_percent_income'] - analysis_train['loan_amnt'] / analysis_train['person_income']).abs()
ratio_summary = pd.DataFrame({
    'Наблюдений': [len(analysis_train)],
    'Расхождений > 0,005': [int((ratio_error > .005 + 1e-9).sum())],
    'Медиана абсолютного расхождения': [ratio_error.median()],
    'Максимальное расхождение': [ratio_error.max()],
})
display(ratio_summary.round(4))


#### 3.4.3 Прошлые дефолты и грейд

Сравним частоты исхода по прошлым дефолтам внутри каждого грейда

In [ ]:
interaction_summary = default_summary(analysis_train, ['grade_ordered', 'cb_person_default_on_file'])
display(group_summary_view(interaction_summary))


#### 3.4.4 Сумма кредита и доход

Сравним частоты дефолта для сочетаний четырёх групп дохода и суммы

In [ ]:
amount_income_summary = default_summary(analysis_train, ['income_quartile', 'amount_quartile'])
display(group_summary_view(amount_income_summary))


#### 3.4.5 Кредитная история при сопоставимом возрасте и грейде

Сравним группы по длине истории внутри возрастных диапазонов и грейдов. В условной сводке оставим группы от 30 записей

In [ ]:
age_history_summary = default_summary(analysis_train, ['age_band', 'history_band'])
age_grade_history_summary = default_summary(analysis_train, ['age_band', 'grade_ordered', 'history_band'])
conditional_history_checks = (age_grade_history_summary.loc[age_grade_history_summary['n'] >= 30]
    .groupby(['age_band', 'grade_ordered'], observed=True)
    .agg(history_groups=('n', 'size'), min_cell_n=('n', 'min'),
         min_default_rate=('default_rate', 'min'), max_default_rate=('default_rate', 'max')))

display(group_summary_view(age_history_summary))
display(group_summary_view(conditional_history_checks))


#### 3.4.6 Высокие доходы

Граница высокого дохода — 95-й процентиль train. Сравним частоты дефолта ниже и выше неё

In [ ]:
income_95 = float(analysis_train['person_income'].quantile(.95))
high_income_summary = default_summary(analysis_train.assign(high_income=analysis_train['person_income'] > income_95), ['high_income'])
display(pd.Series({'Граница высокого дохода': income_95}))
display(group_summary_view(high_income_summary))


#### 3.4.7 Порядок записей в файле

Проверим, меняются ли состав заёмщиков и пропуски между последовательными блоками файла. Номер строки в признаки модели не включаем

In [ ]:
file_summary = analysis_train.groupby('file_block', observed=True).agg(
    n=('loan_status', 'size'), age_median=('person_age', 'median'),
    history_median=('cb_person_cred_hist_length', 'median'), income_median=('person_income', 'median'),
    grade_A_share=('loan_grade', lambda s: s.eq('A').mean()),
    rate_missing=('loan_int_rate', lambda s: s.isna().mean()),
)

display(group_summary_view(file_summary))


#### 3.4.8 Форма связей

Посмотрим на ставки по грейдам и сочетания возраста с длиной кредитной истории

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.boxplot(data=analysis_train, x='grade_ordered', y='loan_int_rate', ax=axes[0], color='#78a6c8', showfliers=False)
axes[0].set(title='Ставка по грейду (train)', xlabel='Грейд A–G', ylabel='Процентная ставка, %')
age_history_counts = pd.crosstab(analysis_train['cb_person_cred_hist_length'], analysis_train['person_age'])
sns.heatmap(np.log1p(age_history_counts), cmap='Blues', ax=axes[1], cbar_kws={'label': 'log(1 + число наблюдений)'})
axes[1].set(title='Возраст и кредитная история (train)', xlabel='Возраст', ylabel='Длина кредитной истории')
plt.tight_layout()
plt.show()


#### Выводы по проверке гипотез

In [ ]:
# include: _code/conclusions.py